# L0 step 1 · Our 7B at the paper's image size (disaster type and relational reasoning)

Task **L0**, step 1, from `Essential_task_delegation/Next_Steps_Roadmap.md` (decided 19 Oct 2026: improve the 7B before distilling). Built as a copy of `h1-v2-counts-as-text.ipynb`, which stays unchanged: the same model, prompts, greedy decoding and scorer. Only the image size changes.

**Why**
- On optical questions our 7B trails the DisasterM3 paper's own fine-tune of the same model (Table 2) by 12.5 points on disaster type (71.1 against 83.6) and 12.9 on relational reasoning (23.3 against 36.2).
- Our training recipe matches the paper's (Appendix B.3), but the image size doesn't:
  - we trained with images capped at about 512 tokens (616 px for a 1024 px image);
  - H1 V2 tested at 512 px (504 px after Qwen's rounding, 324 tokens);
  - the paper tests every model at its default, the full size (1036 px, 1,369 tokens).
- So this run asks whether the gap is in how we test or in the model.

**What it does**
- **Model:** our fine-tuned Qwen2.5-VL-7B, merged (`AbrarAlam/disasterm3-qwen2.5vl7b-mergedFP`), fp16 on two T4s. Greedy decoding, the official prompts and the 499A scorer, as in H1 V2.
- **Questions:** the optical ones of the two tasks, as in the paper's Table 2: 353 disaster type (DTR) and 858 relational reasoning (ORR), 1,211 in all.
- **Three image sizes, in this order:**

| Setting | A 1024 px image reaches the model at | Tokens per image | Why |
|---|---|---|---|
| `512` | 504 px | 324 | H1 V2's setting, re-run as the control, so every change is measured within one run |
| `train` | 616 px | 484 | our training cap (`max_pixels = 512 × 28 × 28`) |
| `full` | 1036 px | 1,369 | Qwen2.5-VL's default, the paper's setting |

- Cell 4 checks each setting's size on a test image before the long run.

**Decision** (fixed before the run; the targets are the paper's fine-tune, 83.6 and 36.2):
- **Gap closed:** the best setting is within 3 points on both tasks. Re-score all six tasks at that setting, then L2 with this 7B.
- **Partly closed:** at least half of each gap recovered. Re-score all six at that setting, then decide on step 2.
- **Not closed:** go to step 2, a short top-up training of the adapter on these two tasks at a larger image size.
- 3 points is about the noise on 353 questions (one standard error is 2.4 points on DTR).

**How to run on Kaggle**
1. Settings: **GPU T4 ×2**, internet ON.
2. Input: `redwannewaz/disasterm3-bench-mirror-v2` (the benchmark JSON and images, found by file name).
3. Secret `HF_TOKEN` (AbrarAlam), optional: the model repo is public.
4. Run with `SMOKE_TEST = True` first: 10 questions per task and setting, about 15 minutes including the 16 GB model download. Then use Save & Run All, about 2–3 hours.
   - Answers are written as they come, and a re-run skips them.
   - The run stops cleanly at 11 hours. To continue, attach that version's output and run again.
5. Outputs in `/kaggle/working/L0S1/`:
   - `l0_step1_results.csv` (accuracy per task and setting), `L0_step1_per_question.csv`, `config.json`, `README.md`;
   - `responses/<setting>/<subset>.jsonl`, with question IDs as in H1 V2.

In [ ]:
# ── Cell 1: Environment (Kaggle's own torch; transformers only if too old for Qwen2.5-VL) ──
import importlib.metadata as _md
import subprocess
import sys

def _version(pkg):
    try:
        return _md.version(pkg)
    except _md.PackageNotFoundError:
        return "0"

def _older(have, want):
    nums = lambda v: tuple(int(x) for x in v.split(".")[:2] if x.isdigit())
    return nums(have) < nums(want)

if _older(_version("transformers"), "4.49") or _version("accelerate") == "0":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers>=4.49", "accelerate"], check=True)

import torch
import transformers
print(f"✓ torch {torch.__version__}, transformers {transformers.__version__}, python {sys.version.split()[0]}")
print(f"  GPUs: {[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}")
assert torch.cuda.device_count() >= 2, "Pick GPU T4 x2: the 7B model in fp16 needs both T4s"

In [ ]:
# ── Cell 2: Configuration, the benchmark's optical questions and their images ──
import glob
import json
import math
import os
import re
import shutil
import time
from collections import Counter
from pathlib import Path

import pandas as pd

MODEL_ID = "AbrarAlam/disasterm3-qwen2.5vl7b-mergedFP"   # our fine-tuned 7B (CSE499A), merged
OUT_DIR = Path("/kaggle/working/L0S1")
SMOKE_TEST = False             # True = 10 questions per task and setting
BATCH = 8                      # questions per generate() call; halved automatically on out-of-memory
MAX_NEW_TOKENS = 32            # answers are option letters
TIME_LIMIT_H = 11.0            # stop cleanly before Kaggle's 12-hour limit
# (code, subset name used by the 499A scorer, task name in the benchmark)
TASKS = [("DTR", "disaster_type", "Disaster Type Recognition"),
         ("ORR", "relational_reasoning_qa", "Relational Reasoning")]
SETTINGS = {"512": "resized to 512 x 512 as in H1 V2; Qwen's rounding then gives 504 px",
            "train": "our training cap (min_pixels 256 x 28 x 28, max_pixels 512 x 28 x 28): 616 px",
            "full": "the native image at Qwen2.5-VL's default, as in the paper: 1036 px"}
EXPECTED_SIDE = {"512": 504, "train": 616, "full": 1036}    # for a 1024 x 1024 image, checked in Cell 4
TRAIN_PIXELS = (256 * 28 * 28, 512 * 28 * 28)               # MIN_PIXELS, MAX_PIXELS in train_qwen_disasterm3.ipynb
REF_PAPER_FT = {"DTR": 83.6, "ORR": 36.2}   # DisasterM3 Table 2 (optical): the authors' fine-tuned Qwen2.5-VL-7B
REF_BASE = {"DTR": 66.6, "ORR": 23.9}       # the same table: the base Qwen2.5-VL-7B
REF_H1V2 = {"DTR": 71.1, "ORR": 23.3}       # our 7B at 512 px on the same optical questions (H1 V2)
EXPECTED_QUESTIONS = {"DTR": 353, "ORR": 858}

def hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        return os.environ.get("HF_TOKEN")

def find_one(pattern, why):
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    if not hits:
        raise FileNotFoundError(f"No {pattern} under /kaggle/input: {why}")
    return Path(hits[0])

def name_of(path):
    return Path(str(path).replace("\\", "/")).name

def sensor(e):
    """H1 V2's rule for the after-image's sensor: DTR entries have post_image_type, ORR entries image_type."""
    return e.get("post_image_type") or e.get("image_type") or "Optical"

with open(find_one("benchmark_release.json", "attach redwannewaz/disasterm3-bench-mirror-v2"), encoding="utf-8") as f:
    bench = json.load(f)

bench_index = {}                # every benchmark image by file name: mirrors nest them differently
for p in glob.glob("/kaggle/input/**/test_images/**/*.png", recursive=True):
    bench_index.setdefault(Path(p).name, p)

questions = {}                  # code -> [(id as in H1 V2 and the 499A scorer, entry)], optical only
for code, subset, task in TASKS:
    qs = [(f"{subset}_{i}", e) for i, e in enumerate(e for e in bench if e.get("task") == task)]
    qs = [(qid, e) for qid, e in qs if sensor(e) == "Optical"]
    questions[code] = qs[:10] if SMOKE_TEST else qs
if not SMOKE_TEST:
    assert {c: len(q) for c, q in questions.items()} == EXPECTED_QUESTIONS, "not the optical questions H1 V2 scored"

needed = {name_of(e[k]) for qs in questions.values() for _, e in qs
          for k in ("pre_image_path", "post_image_path", "image_path") if e.get(k)}
found = sum(n in bench_index for n in needed)
if found < 0.99 * len(needed):
    raise FileNotFoundError(f"Only {found:,} of {len(needed):,} benchmark images found: attach the bench mirror.")

OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"✓ L0 step 1{'  [SMOKE TEST]' if SMOKE_TEST else ''}: {MODEL_ID}")
print(f"  Optical questions: {({c: len(q) for c, q in questions.items()})}; settings: {list(SETTINGS)}")
print(f"  Images found: {found:,} of {len(needed):,}")

In [ ]:
# ── Cell 3: The official prompts (H1 V2's), the three image sizes and the 499A scorer ──
from PIL import Image

# Copied from H1 V2, which copied them from temp_junjue/pyscripts/run_vllm.py (the DisasterM3 authors' runner)
PROMPTS = dict(
    single_choice="""Analyze both the pre-disaster and post-disaster images to answer the following question. Choose the best option from the candidate options provided.

pre-disaster image:
<image>

post-disaster image:
<image>

Question: {prompt}
Options: {options_str}

Your task is to respond with ONLY the capital letter of the correct option (e.g., C). Do not include any explanation or other text.""",
    relational_reasoning_qa="""Analyze the image to answer the following question. Choose the best option from the candidate options provided.

Question: {prompt}
Options: {options_str}

Your task is to respond with ONLY the capital letter of the correct option (e.g., C). Do not include any explanation or other text.""",
)

def smart_resize(height, width, min_pixels, max_pixels, factor=28):
    """Qwen2.5-VL's rule for the size an image reaches the model at (its image processor's smart_resize)."""
    h, w = round(height / factor) * factor, round(width / factor) * factor
    if h * w > max_pixels:
        beta = math.sqrt(height * width / max_pixels)
        h, w = math.floor(height / beta / factor) * factor, math.floor(width / beta / factor) * factor
    elif h * w < min_pixels:
        beta = math.sqrt(min_pixels / (height * width))
        h, w = math.ceil(height * beta / factor) * factor, math.ceil(width * beta / factor) * factor
    return h, w

def sized(img, setting):
    """The image as this setting hands it to the processor, which keeps Qwen's default limits."""
    if setting == "512":
        return img.resize((512, 512), Image.BICUBIC)            # exactly as H1 V2
    if setting == "train":                                      # as the training processor resized it
        h, w = smart_resize(img.height, img.width, *TRAIN_PIXELS)
        return img.resize((w, h), Image.BICUBIC)
    return img                                                  # "full": the native image

def build(e, subset, setting):
    """-> (chat messages, PIL images in prompt order), built as the official runner builds them."""
    if subset == "relational_reasoning_qa":     # this task's entries spell it option_str
        text, paths = PROMPTS[subset].format(prompt=e["prompts"], options_str=e["option_str"]), [e["image_path"]]
    else:
        text = PROMPTS["single_choice"].format(prompt=e["prompts"], options_str=e["options_str"])
        paths = [e["pre_image_path"], e["post_image_path"]]
    parts = text.split("<image>")
    if len(parts) == 1:                            # single-image templates: the image comes first
        content = [{"type": "image"}, {"type": "text", "text": text}]
    else:
        content = [{"type": "text", "text": parts[0]}, {"type": "image"}, {"type": "text", "text": parts[1]},
                   {"type": "image"}, {"type": "text", "text": parts[2]}]
    images = [sized(Image.open(bench_index[name_of(p)]).convert("RGB"), setting) for p in paths]
    return [{"role": "user", "content": content}], images

def letters(text):
    """The 499A scorer (4_score_results.py): the set of capital letters left after removing
    'THE ANSWER IS' and 'OPTION'. An answer counts only if this set equals the truth's set."""
    t = str(text).upper().replace("THE ANSWER IS", "").replace("OPTION", "")
    return set(re.findall(r"[A-Z]", t))

# Self-checks: the scorer, Qwen's rounding, and the number of images per prompt
assert letters("B, E, G") == {"B", "E", "G"} and letters("The answer is D.") == {"D"} and letters("") == set()
assert smart_resize(1024, 1024, *TRAIN_PIXELS) == (616, 616) and smart_resize(512, 512, 3136, 12845056) == (504, 504)
assert len(build(questions["DTR"][0][1], "disaster_type", "512")[1]) == 2
assert len(build(questions["ORR"][0][1], "relational_reasoning_qa", "full")[1]) == 1
native = Counter(Image.open(bench_index[name_of(e.get("image_path") or e["post_image_path"])]).size
                 for qs in questions.values() for _, e in qs[:50])
print(f"✓ Prompts ready (self-checks passed). Native sizes of the first 50 after-images per task: {dict(native)}")

In [ ]:
# ── Cell 4: Load the model (fp16, both T4s), check each image size, and the batched greedy generator ──
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

t0 = time.time()
processor = AutoProcessor.from_pretrained(MODEL_ID, token=hf_token())
processor.tokenizer.padding_side = "left"              # batched generation needs left padding
# transformers 4.56 renamed torch_dtype to dtype; older versions may ignore dtype and load fp32 (30 GB)
dtype_arg = "torch_dtype" if _older(transformers.__version__, "4.56") else "dtype"
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID, **{dtype_arg: torch.float16}, device_map="auto", attn_implementation="sdpa", token=hf_token())
model.eval()

# Each setting on a 1024 x 1024 test image: the side the model sees, from the processor's grid of 14 px patches.
# The merged repo saved the stock Qwen2.5-VL processor (1_merge_and_push_model.ipynb), so "full" stays full size.
_test = Image.new("RGB", (1024, 1024), (90, 120, 80))
for setting, want in EXPECTED_SIDE.items():
    _, h, w = processor.image_processor(images=[sized(_test, setting)], return_tensors="pt")["image_grid_thw"][0].tolist()
    assert (h * 14, w * 14) == (want, want), (f"'{setting}': the model would see {h * 14} x {w * 14} px, not {want}. "
                                              "Check the processor's min_pixels and max_pixels.")
    print(f"  {setting:>5}: a 1024 px image reaches the model at {want} px ({h * w // 4:,} tokens)")

visual_tokens, asked = Counter(), Counter()     # image tokens fed and questions asked, per setting

@torch.inference_mode()
def generate(batch, setting):
    """batch: list of (messages, images) -> list of answers. Greedy, so a re-run gives the same answers."""
    texts = [processor.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m, _ in batch]
    images = [img for _, imgs in batch for img in imgs]
    inputs = processor(text=texts, images=images, padding=True, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, temperature=None,
                         top_p=None, top_k=None, repetition_penalty=1.0)
    visual_tokens[setting] += int(inputs["image_grid_thw"].prod(1).sum()) // 4
    asked[setting] += len(batch)
    new_tokens = out[:, inputs["input_ids"].shape[1]:]
    return [s.strip() for s in processor.batch_decode(new_tokens, skip_special_tokens=True)]

def generate_safe(batch, setting):
    """Halves the batch on out-of-memory instead of crashing the run."""
    try:
        return generate(batch, setting)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        if len(batch) == 1:
            return [""]
        half = len(batch) // 2
        return generate_safe(batch[:half], setting) + generate_safe(batch[half:], setting)

# One question of each task at each setting, before the long run
for setting in SETTINGS:
    for code, subset, _ in TASKS:
        e = questions[code][0][1]
        print(f"  {setting:>5} {code}: truth {e['ground_truth_option']!r} | "
              f"answer {generate_safe([build(e, subset, setting)], setting)[0]!r}")
visual_tokens.clear()
asked.clear()
print(f"✓ Model loaded ({dtype_arg}=float16) and checked in {(time.time() - t0) / 60:.1f} min; "
      f"GPU memory {[f'{torch.cuda.memory_allocated(i) / 1e9:.1f} GB' for i in range(torch.cuda.device_count())]}")

## Run: every question at each setting
Settings run in the order `512`, `train`, `full`, the cheapest first, so if the 11-hour limit stops the run, the finished settings are complete. Answers are appended to `responses/<setting>/<subset>.jsonl` after every batch; a re-run (or a new version with this one's output attached) skips the questions already answered.

In [ ]:
# ── Cell 5: Ask every question at each setting, writing answers as they come ──
from tqdm.auto import tqdm

# Continue an earlier run: copy its answers in when that run's output is attached
previous = sorted(glob.glob("/kaggle/input/**/L0S1/responses", recursive=True))
if previous and not (OUT_DIR / "responses").exists():
    shutil.copytree(previous[0], OUT_DIR / "responses")
    print(f"Continuing from {previous[0]}")

t_start = time.time()
stopped, unreadable, minutes = False, Counter(), Counter()
for setting in SETTINGS:
    for code, subset, _ in TASKS:
        path = OUT_DIR / "responses" / setting / f"{subset}.jsonl"
        path.parent.mkdir(parents=True, exist_ok=True)
        done = {json.loads(line)["id"] for line in open(path, encoding="utf-8")} if path.exists() else set()
        todo = [(qid, e) for qid, e in questions[code] if qid not in done]
        t_task = time.time()
        with open(path, "a", encoding="utf-8") as f:
            for i in tqdm(range(0, len(todo), BATCH), desc=f"{setting} {code}"):
                if (time.time() - t_start) / 3600 > TIME_LIMIT_H:
                    stopped = True
                    break
                built = []
                for qid, e in todo[i:i + BATCH]:
                    try:
                        built.append((qid, build(e, subset, setting)))
                    except (KeyError, OSError):              # image missing or unreadable: no answer, scored wrong
                        unreadable[code] += 1
                        f.write(json.dumps({"id": qid, "response": ""}) + "\n")
                answers = generate_safe([b for _, b in built], setting) if built else []
                for (qid, _), answer in zip(built, answers):
                    f.write(json.dumps({"id": qid, "response": answer}) + "\n")
                f.flush()
        minutes[f"{setting} {code}"] = round((time.time() - t_task) / 60, 1)
        if stopped:
            break
    if stopped:
        break
run_hours = (time.time() - t_start) / 3600
print(f"{'⚠ Stopped at the time limit' if stopped else '✓ All questions answered'} after {run_hours:.1f} h; "
      f"minutes per setting and task: {dict(minutes)}; unreadable images: {dict(unreadable) or 'none'}")

In [ ]:
# ── Cell 6: Score each setting, the verdict, and the output files ──────────
answers = {}
for setting in SETTINGS:
    for path in (OUT_DIR / "responses" / setting).glob("*.jsonl"):
        for line in open(path, encoding="utf-8"):
            r = json.loads(line)
            answers[(setting, r["id"])] = r["response"]

rows = []
for code, _, _ in TASKS:
    for qid, e in questions[code]:
        row = {"task": code, "id": qid, "truth": e["ground_truth_option"]}
        for setting in SETTINGS:
            if (setting, qid) in answers:
                row[setting] = answers[(setting, qid)]
                row[f"{setting}_correct"] = letters(answers[(setting, qid)]) == letters(e["ground_truth_option"])
        rows.append(row)
per_q = pd.DataFrame(rows)
per_q.to_csv(OUT_DIR / "L0_step1_per_question.csv", index=False)

table = []
for code, _, _ in TASKS:
    df = per_q[per_q["task"] == code]
    row = {"task": code, "questions": len(df)}
    for setting in SETTINGS:
        col = f"{setting}_correct"
        answered = df[col].dropna() if col in df else pd.Series(dtype=bool)
        row[f"{setting}_%"] = round(100 * answered.astype(bool).mean(), 2) if len(answered) else float("nan")
        row[f"{setting}_answered"] = len(answered)
    row.update({"h1v2_512_%": REF_H1V2[code], "paper_finetuned_%": REF_PAPER_FT[code], "paper_base_%": REF_BASE[code]})
    table.append(row)
results = pd.DataFrame(table)
results.to_csv(OUT_DIR / "l0_step1_results.csv", index=False)
print(results.to_string(index=False))

# The verdict, by the rule fixed in the first cell, from the settings that answered every question
by_task = {r["task"]: r for r in table}
complete = [s for s in SETTINGS if all(r[f"{s}_answered"] == r["questions"] for r in table)]
best, verdict = None, "n/a (smoke test)" if SMOKE_TEST else "incomplete: attach this output and run again"
if not SMOKE_TEST and "512" in complete and len(complete) > 1:
    best = max(complete, key=lambda s: sum(r[f"{s}_%"] for r in table))
    gap = {c: REF_PAPER_FT[c] - by_task[c]["512_%"] for c in by_task}          # from this run's control
    left = {c: REF_PAPER_FT[c] - by_task[c][f"{best}_%"] for c in by_task}
    recovered = {c: 1.0 if gap[c] <= 0 else (gap[c] - left[c]) / gap[c] for c in by_task}
    shown = ", ".join(f"{c} {by_task[c][f'{best}_%']:.1f}% ({recovered[c]:.0%} of the gap)" for c in by_task)
    if all(v <= 3 for v in left.values()):
        verdict = (f"Gap closed at '{best}' ({shown}): within 3 points of the paper's fine-tune on both tasks. "
                   f"Next: re-score all six tasks at '{best}', then L2 with this 7B.")
    elif all(v >= 0.5 for v in recovered.values()):
        verdict = (f"Gap partly closed at '{best}' ({shown}). "
                   f"Next: re-score all six tasks at '{best}', then decide on step 2.")
    else:
        verdict = (f"Gap not closed: the best setting is '{best}' ({shown}). "
                   "The image size isn't the main cause; next is step 2, a top-up training of the adapter.")
print(f"\n{verdict}")
tokens = {s: round(visual_tokens[s] / asked[s]) for s in SETTINGS if asked[s]}
print(f"Image tokens per question as fed (this session): {tokens}")

def as_markdown(df):
    try:
        return df.to_markdown(index=False)
    except ImportError:              # `tabulate` missing: fall back to a plain-text table
        return "```\n" + df.to_string(index=False) + "\n```"

config = {"task": "L0 step 1", "smoke_test": SMOKE_TEST, "model": MODEL_ID, "decoding": "greedy",
          "settings": SETTINGS, "expected_side_for_1024px": EXPECTED_SIDE, "image_tokens_per_question": tokens,
          "questions": {c: len(q) for c, q in questions.items()}, "batch": BATCH, "max_new_tokens": MAX_NEW_TOKENS,
          "transformers": transformers.__version__, "torch": torch.__version__, "stopped_at_time_limit": stopped,
          "run_hours": round(run_hours, 2), "minutes": dict(minutes), "unreadable_images": dict(unreadable),
          "complete_settings": complete, "best_setting": best, "verdict": verdict}
(OUT_DIR / "config.json").write_text(json.dumps(config, indent=1), encoding="utf-8")
(OUT_DIR / "README.md").write_text("\n".join([
    "# L0 step 1: our 7B at three image sizes", "",
    f"Model `{MODEL_ID}`, greedy decoding, the official DisasterM3 prompts, optical questions only "
    f"({', '.join(f'{c} {len(q)}' for c, q in questions.items())}){' (SMOKE TEST)' if SMOKE_TEST else ''}. "
    "`512` is H1 V2's setting, re-run here as the control.", "",
    "## Accuracy (%)", "", as_markdown(results), "",
    f"**Verdict:** {verdict}", "",
    "## Settings", "",
    *[f"- `{s}`: {d}; {tokens.get(s, 'n/a')} image tokens per question as fed" for s, d in SETTINGS.items()], "",
    "## Notes", "",
    "- The references come from DisasterM3's Table 2 (optical questions): the base Qwen2.5-VL-7B and the authors' "
    "fine-tune of it, both run at Qwen's default image size.",
    "- The decision rule was fixed before the run (first cell): within 3 points of the paper on both tasks closes "
    "the gap; half of each gap recovered closes it partly.", "",
    "## Files", "",
    "- `l0_step1_results.csv`, `L0_step1_per_question.csv`, `config.json`",
    "- `responses/<setting>/<subset>.jsonl`, with question IDs as in H1 V2",
]), encoding="utf-8")
print(f"✓ Wrote l0_step1_results.csv, L0_step1_per_question.csv, README.md and config.json to {OUT_DIR}")